# 09 · Transactions

Service Bus supports **atomic groups** of operations against a single entity (or, with **transfer queues**, across two). You wrap the operations in a `TransactionScope`; everything inside either all commits or all rolls back.

Common pattern: **receive → process → send result → complete**, all atomically, so a crash mid-way doesn't lose or duplicate work.

> Requires `System.Transactions`. Service Bus does **not** participate in distributed two-phase commits — transactions stay within Service Bus.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;
using System.Transactions;

// EnableCrossEntityTransactions = true is required when the transaction spans
// more than one entity (e.g. receive from queue, send to topic).
var client = new ServiceBusClient(Config.ConnectionString, new ServiceBusClientOptions
{
    EnableCrossEntityTransactions = true
});

var sender   = client.CreateSender(Config.QueueName);
var receiver = client.CreateReceiver(Config.QueueName);

await sender.SendMessageAsync(new ServiceBusMessage("input-A"));

## 1. Atomic 'process then forward'

In [ ]:
var msg = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"Received: {msg.Body}");

using (var scope = new TransactionScope(TransactionScopeAsyncFlowOption.Enabled))
{
    // pretend to compute something based on the message
    await sender.SendMessageAsync(new ServiceBusMessage($"processed:{msg.Body}"));
    await receiver.CompleteMessageAsync(msg);

    scope.Complete(); // commit both
}
Console.WriteLine("Committed.");

var forwarded = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"Forwarded message visible: {forwarded.Body}");
await receiver.CompleteMessageAsync(forwarded);

## 2. Rollback example

If we don't call `scope.Complete()`, both the send and the complete are undone.

In [ ]:
await sender.SendMessageAsync(new ServiceBusMessage("input-B"));
var m = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));

try
{
    using var scope = new TransactionScope(TransactionScopeAsyncFlowOption.Enabled);
    await sender.SendMessageAsync(new ServiceBusMessage("shouldnt-exist"));
    await receiver.CompleteMessageAsync(m);
    throw new Exception("simulated failure before scope.Complete()");
    // scope.Complete();   // never called
}
catch (Exception ex)
{
    Console.WriteLine($"Caught: {ex.Message}  → both ops rolled back.");
}

// 'm' should be available again after lock expiry
Console.WriteLine("input-B remains for redelivery (after its lock expires).");

In [ ]:
await sender.DisposeAsync();
await receiver.DisposeAsync();
await client.DisposeAsync();

Next: [`10-managed-identity.ipynb`](10-managed-identity.ipynb)